In [5]:
import numpy as np
import tiktoken
import torch 
import math
import torch.nn as nn
import torch.nn.functional as F


class TokenEmbeddings(nn.Module):
    
    def __init__(self,vocab_size,num_dims):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(vocab_size,num_dims)*0.02)

    def forward(self,indices):
        return self.weights[indices]


class PositionalEmbeddings(nn.Module):
    
    def __init__(self,seq_len,num_dims):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(seq_len,num_dims)*0.02)

    def forward(self,positions):
        return self.weights[positions]


class LinearLayer(nn.Module):
    def __init__(self,in_features,out_features):
        super().__init__()
        kaim = math.sqrt(2/in_features)
        self.weights = nn.Parameter(torch.randn(out_features,in_features)*kaim)
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self,x):
        return x @ self.weights.T + self.bias


class Relu(nn.Module):
    
    def __init__(self):
        super().__init__()

    def forward(self,x):
        return torch.clamp(x,min=0)


class Dropout(nn.Module):
    
    def __init__(self,p=0.1):
        super().__init__()
        self.p = p


    def forward(self,x):
        if self.training and self.p>0.0:
            self.mask = (torch.rand_like(x) > self.p).float()/(1.0-self.p)
            return self.mask *x
        return x

class MyMlp(nn.Module):
    
    def __init__(self,in_features,hidden_features,out_features):
        super().__init__()
        self.linear1 = LinearLayer(in_features,hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features,out_features)
        self.dropout = Dropout(p=0.1)

    def forward(self,x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = self.dropout(x)

        return x
        
class CausalSelfAttention(nn.Module):
    
    def __init__(self,num_dims,num_heads):
        super().__init__()
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = nn.Linear(num_dims,num_dims,bias=False)
        self.w_k = nn.Linear(num_dims,num_dims,bias=False)
        self.w_v = nn.Linear(num_dims,num_dims,bias=False)

        self.proj_out = nn.Linear(num_dims,num_dims)
        self.attn_dropout  = Dropout(p=0.1)
        self.resid_dropout = Dropout(p=0.1)

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q(x)
        K = self.w_k(x)
        V = self.w_v(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)
        scores = scores.masked_fill(masks,float("-inf"))
        scores = F.softmax(scores,dim=-1)

        attn_scores = self.attn_dropout(scores)

        out = attn_scores @ V
        out = out.transpose(1,2).contiguous().view(B,T,D)

        out = self.proj_out(out)
        out = self.resid_dropout(out)
        return out

class LayerNorm(nn.Module):
    
    def __init__(self, num_dims, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.num_dims = num_dims
        self.gamma = nn.Parameter(torch.ones(num_dims))
        self.beta = nn.Parameter(torch.zeros(num_dims))

    def forward(self, x):
        avg = torch.mean(x, dim=-1, keepdim=True)
        var = torch.mean((x - avg) ** 2, dim=-1, keepdim=True)
        x_center = x - avg
        inv_std = 1.0 / torch.sqrt(var + self.eps)

        x_norm = (x_center * inv_std) * self.gamma + self.beta
        return x_norm


class Transfomer(nn.Module):
    
    def __init__(self,num_dims,num_heads):
        super().__init__()
        self.ln_1 = LayerNorm(num_dims)
        self.attn = CausalSelfAttention(num_dims,num_heads)
        self.ln_2 = LayerNorm(num_dims)
        self.mlp = MyMlp(num_dims,4*num_dims,num_dims)

    def forward(self,x):
        x = x+ self.attn(self.ln_1(x))

        x = x+self.mlp(self.ln_2(x))

        return x


class GPT(nn.Module):

    def __init__(self,vocab_size,seq_len,num_dims,num_heads,num_layers):
        super().__init__()

        self.seq_len = seq_len
        self.tok_emb = TokenEmbeddings(vocab_size,num_dims)
        self.pos_emb = PositionalEmbeddings(seq_len,num_dims)
        self.blocks = nn.ModuleList([Transfomer(num_dims,num_heads) for _ in range(num_layers)])
        self.ln_f = LayerNorm(num_dims,eps=1e-5)
        self.lm_head = nn.Linear(num_dims,vocab_size,bias=False)

    def forward(self,idx,targets=None):
        B,T = idx.shape
        positions = torch.arange(0,T,dtype=torch.long, device=idx.device)

        tok = self.tok_emb(idx)
        pos = self.pos_emb(positions)
        x = tok+pos

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            logits_flat = logits.view(B*T,vocab_size)
            targets_flat = targets.view(B*T,)
            loss = F.cross_entropy(logits_flat,targets_flat)

        return logits,loss
        

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab_size=50257
model = GPT(  vocab_size=50257, seq_len=64, num_dims=128, num_heads=4, num_layers=4).to(device)
        
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)


train_data = np.memmap("./sample_dataset/train.bin",dtype=np.uint16,mode="r")
val_data = np.memmap("./sample_dataset/val.bin",dtype=np.uint16,mode="r")

def get_batches(split="train",batch_size=4,block_size=64,device="cpu"):

    # select the data source
    d = train_data if split=="train" else val_data

    # finding the index
    ix = torch.randint(0,len(d)-block_size,(batch_size,))

    # slicing x and y
    x_list = [torch.from_numpy((d[i:i+block_size]).astype(np.int64)) for i in ix]
    y_list = [torch.from_numpy((d[i+1:i+1+block_size]).astype(np.int64)) for i in ix]


    # stack the individual 1D to the 2D (B,T)
    x = torch.stack(x_list)
    y = torch.stack(y_list)

    if device != "cpu":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
        
    return x, y


xb, yb = get_batches(split="train", batch_size=4, block_size=64, device=device)

max_iters = 100

for i in range(max_iters):
    # 1. Fetch fresh batch
    xb, yb = get_batches(split="train", batch_size=4, block_size=64, device=device)

    # 2. Clear old gradients
    optimizer.zero_grad(set_to_none=True)

    # 3. Forward pass
    logits, loss = model(xb, yb)

    # 4. Backward pass
    loss.backward()

    # 5. Clip gradients (safety net)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # 6. Update weights
    optimizer.step()

    if i % 10 == 0:
        print(f"Step {i:3d} | Loss: {loss.item():.4f}")

Step   0 | Loss: 10.9525
Step  10 | Loss: 9.8861
Step  20 | Loss: 8.3611
Step  30 | Loss: 7.5831
Step  40 | Loss: 7.2174
Step  50 | Loss: 7.2258
Step  60 | Loss: 7.2720
Step  70 | Loss: 7.3648
Step  80 | Loss: 7.7047
Step  90 | Loss: 7.4012
